# 3.2 Software pipeline results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from style import PRIMARY_COLOR, SECONDARY_COLOR, finish_axis, save_figure

In [ ]:
PIPELINE_DIR = Path.home() / "Desktop" / "dvc" / "pipeline"

Load subject- and recording-level spectral matrices (PhysioNet/CinC 2016, Training set b).

In [ ]:
def load_npz(path):
    with np.load(path, allow_pickle=True) as data:
        return {key: data[key] for key in data.files}


def frequency_vector(data):
    for key in ["freq_hz", "frequency_hz", "freq"]:
        if key in data:
            return np.asarray(data[key])
    raise KeyError("No frequency vector found.")


subject_normal = load_npz(
    PIPELINE_DIR / "group_matrices_subject" / "normal_matrix_subject.npz"
)
subject_cad = load_npz(PIPELINE_DIR / "group_matrices_subject" / "cad_matrix_subject.npz")
recording_normal = load_npz(
    PIPELINE_DIR / "group_matrices_recording" / "normal_matrix_recording.npz"
)
recording_cad = load_npz(
    PIPELINE_DIR / "group_matrices_recording" / "cad_matrix_recording.npz"
)

freq_subject = frequency_vector(subject_normal)
freq_recording = frequency_vector(recording_normal)

Xn_subject = np.asarray(subject_normal["X"])
Xc_subject = np.asarray(subject_cad["X"])
Xn_recording = np.asarray(recording_normal["X"])
Xc_recording = np.asarray(recording_cad["X"])

print("Subject:", Xn_subject.shape, Xc_subject.shape)
print("Recording:", Xn_recording.shape, Xc_recording.shape)

#### Figure 11 — Mean spectral profiles for Normal and CAD groups at subject and recording levels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6), sharey=True)

for ax, frequencies, normal, cad, title in [
    (axes[0], freq_subject, Xn_subject, Xc_subject, "(a) Subject level"),
    (axes[1], freq_recording, Xn_recording, Xc_recording, "(b) Recording level"),
]:
    mask = frequencies <= 300
    f = frequencies[mask]

    normal_mean = normal[:, mask].mean(axis=0)
    normal_sd = normal[:, mask].std(axis=0, ddof=1)
    cad_mean = cad[:, mask].mean(axis=0)
    cad_sd = cad[:, mask].std(axis=0, ddof=1)

    ax.plot(f, normal_mean, color=PRIMARY_COLOR, label=f"Normal (n = {len(normal):,})")
    ax.fill_between(
        f,
        normal_mean - normal_sd,
        normal_mean + normal_sd,
        color=PRIMARY_COLOR,
        alpha=0.16,
        linewidth=0,
    )

    ax.plot(f, cad_mean, color=SECONDARY_COLOR, label=f"CAD (n = {len(cad):,})")
    ax.fill_between(
        f, cad_mean - cad_sd, cad_mean + cad_sd, color=SECONDARY_COLOR, alpha=0.16, linewidth=0
    )

    ax.set_title(title, loc="left", fontsize=13, fontweight="semibold", pad=7)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Average spectral amplitude (a.u.)")
    ax.set_xlim(0, 300)
    ax.legend(frameon=False, loc="upper right")
    finish_axis(ax)

fig.tight_layout()
save_figure(fig, "physionet_mean_spectra_subject_recording")
plt.show()

#### Figure 12 — Welch's t-test results comparing Normal and CAD spectral profiles

In [ ]:
def load_ttest(level):
    path = PIPELINE_DIR / f"ttest_{level}" / "ttest_results.npz"

    with np.load(path, allow_pickle=True) as data:
        return {key: data[key] for key in data.files}


subject_ttest = load_ttest("subject")
recording_ttest = load_ttest("recording")

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.9))

plot_configs = [
    {
        "ax": axes[0],
        "result": subject_ttest,
        "panel_label": "(a)",
        "level_label": "Subject-level",
    },
    {
        "ax": axes[1],
        "result": recording_ttest,
        "panel_label": "(b)",
        "level_label": "Recording-level",
    },
]

shared_legend_handles = None
shared_legend_labels = None

for panel_index, config in enumerate(plot_configs):

    ax = config["ax"]
    result = config["result"]

    frequencies = np.asarray(result["freq_hz"], dtype=float)

    p_values = np.asarray(result["p_val"], dtype=float)

    t_values = np.abs(np.asarray(result["t_stat"], dtype=float))

    significant = np.asarray(result["sig"]).astype(bool)

    n_normal = int(np.asarray(result["n_normal"]).item())

    n_cad = int(np.asarray(result["n_cad"]).item())

    frequency_mask = (
        np.isfinite(frequencies)
        & np.isfinite(p_values)
        & np.isfinite(t_values)
        & (frequencies >= 0)
        & (frequencies <= 300)
    )

    frequencies = frequencies[frequency_mask]
    p_values = p_values[frequency_mask]
    t_values = t_values[frequency_mask]
    significant = significant[frequency_mask]

    ax_right = ax.twinx()

    p_line = ax.plot(
        frequencies, p_values, color=PRIMARY_COLOR, linewidth=1.8, label="p-value", zorder=3
    )[0]

    alpha_line = ax.axhline(
        0.05,
        color=SECONDARY_COLOR,
        linestyle="--",
        linewidth=1.5,
        label=r"$\alpha$ = 0.05",
        zorder=2,
    )

    significant_patch = ax.fill_between(
        frequencies,
        0,
        1,
        where=significant,
        color=SECONDARY_COLOR,
        alpha=0.10,
        linewidth=0,
        label=r"$p < \alpha$ (0.05)",
        zorder=0,
    )

    significant_points = ax.scatter(
        frequencies[significant],
        p_values[significant],
        color=SECONDARY_COLOR,
        s=20,
        linewidths=0,
        label="Significant points",
        zorder=5,
    )

    t_line = ax_right.plot(
        frequencies,
        t_values,
        color=SECONDARY_COLOR,
        linewidth=1.8,
        alpha=0.95,
        label=r"Absolute t-statistic, $|t|$",
        zorder=3,
    )[0]

    ax.set_title(
        (
            f"{config['panel_label']} "
            f"{config['level_label']} t-test (per frequency bin)\n"
            f"Normal: {n_normal:,}, CAD: {n_cad:,}"
        ),
        loc="left",
        fontsize=13,
        fontweight="semibold",
        pad=7,
    )

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("p-value")
    ax_right.set_ylabel(r"Absolute t-statistic, $|t|$")

    ax.set_xlim(0, 300)
    ax.set_ylim(0, 1.05)
    ax.set_xticks(np.arange(0, 301, 50))
    ax.set_yticks(np.arange(0, 1.01, 0.2))

    finish_axis(ax, grid_axis="both")

    ax_right.spines["top"].set_visible(False)
    ax_right.grid(False)

    if panel_index == 0:
        shared_legend_handles = [
            p_line,
            alpha_line,
            significant_patch,
            significant_points,
            t_line,
        ]
        shared_legend_labels = [
            "p-value",
            r"$\alpha$ = 0.05",
            r"$p < \alpha$ (0.05)",
            "Significant points",
            r"Absolute t-statistic, $|t|$",
        ]

fig.legend(
    shared_legend_handles,
    shared_legend_labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=5,
    frameon=False,
    fontsize=10,
    handlelength=2.2,
    handletextpad=0.6,
    columnspacing=1.5,
)

fig.subplots_adjust(left=0.08, right=0.94, bottom=0.22, top=0.86, wspace=0.30)

save_figure(fig, "physionet_ttests_subject_recording")

plt.show()

#### Figure 13 — PCA projection of the spectral profiles at subject and recording levels

In [ ]:
def calculate_pca(normal, cad):
    X = np.vstack([normal, cad])
    labels = np.array(["Normal"] * len(normal) + ["CAD"] * len(cad))

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=2)
    scores = pca.fit_transform(X_scaled)

    return scores, labels, pca.explained_variance_ratio_


subject_scores, subject_labels, subject_variance = calculate_pca(Xn_subject, Xc_subject)
recording_scores, recording_labels, recording_variance = calculate_pca(
    Xn_recording, Xc_recording
)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6))

for ax, scores, labels, variance, title in [
    (
        axes[0],
        subject_scores,
        subject_labels,
        subject_variance,
        f"(a) Subject-level PCA (n = {len(subject_labels)}: "
        f"{(subject_labels == 'Normal').sum()} Normal, "
        f"{(subject_labels == 'CAD').sum()} CAD)",
    ),
    (
        axes[1],
        recording_scores,
        recording_labels,
        recording_variance,
        f"(b) Recording-level PCA (n = {len(recording_labels)}: "
        f"{(recording_labels == 'Normal').sum()} Normal, "
        f"{(recording_labels == 'CAD').sum()} CAD)",
    ),
]:
    for diagnosis, color in [("Normal", PRIMARY_COLOR), ("CAD", SECONDARY_COLOR)]:
        mask = labels == diagnosis
        ax.scatter(
            scores[mask, 0],
            scores[mask, 1],
            s=28 if len(scores) < 500 else 16,
            color=color,
            alpha=0.75,
            linewidths=0,
            label=diagnosis,
        )

    ax.set_title(title, loc="left", fontsize=13, fontweight="semibold", pad=7)
    ax.set_xlabel(f"PC1 ({100*variance[0]:.1f}%)")
    ax.set_ylabel(f"PC2 ({100*variance[1]:.1f}%)")
    ax.legend(frameon=False)
    finish_axis(ax)

fig.tight_layout()
save_figure(fig, "physionet_pca_subject_recording")
plt.show()

#### Figure 14 — Within-subject similarity of repeated recordings

In [ ]:
similarity_summary = pd.read_csv(
    PIPELINE_DIR / "within_subject_similarity" / "subject_similarity_summary.csv"
)

qq_summary = pd.read_csv(
    PIPELINE_DIR / "qq_within_subject_recordings" / "subject_qq_summary.csv"
)

within_subject = (
    similarity_summary[["subject_id", "mean_pearson_r"]]
    .merge(
        qq_summary[["subject_id", "mean_spectrum_qq_mae", "mean_spectrum_ks_stat"]],
        on="subject_id",
        how="inner",
        validate="one_to_one",
    )
    .dropna()
)

assert len(within_subject) == 101

plot_config = [
    {
        "column": "mean_pearson_r",
        "title": "(a) Mean Pearson correlation",
        "xlabel": "Mean Pearson correlation",
        "xlim": (0.93, 1.00),
        "xticks": [0.93, 0.95, 0.97, 0.99],
    },
    {
        "column": "mean_spectrum_qq_mae",
        "title": "(b) Mean absolute error",
        "xlabel": ("Mean absolute error between corresponding " "quantiles"),
        "xlim": (0.00, 0.09),
        "xticks": np.arange(0.00, 0.091, 0.01),
    },
    {
        "column": "mean_spectrum_ks_stat",
        "title": "(c) Kolmogorov–Smirnov statistic",
        "xlabel": "Mean Kolmogorov–Smirnov statistic",
        "xlim": (0.00, 0.55),
        "xticks": np.arange(0.0, 0.51, 0.1),
    },
]

fig, axes = plt.subplots(1, 3, figsize=(17.7, 4.6), sharey=True)

for ax, config in zip(axes, plot_config):

    values = pd.to_numeric(within_subject[config["column"]], errors="coerce").dropna()

    mean_value = values.mean()

    ax.hist(values, bins=16, color=PRIMARY_COLOR, alpha=0.88, linewidth=0)

    ax.axvline(mean_value, color=SECONDARY_COLOR, linestyle="--", linewidth=1.8)

    ax.text(
        0.03,
        0.96,
        f"Mean = {mean_value:.3f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=11,
    )

    ax.set_title(
        (
            f"{config['title']}\n"
            f"Subjects with repeated recordings only "
            f"(n = {len(values)})"
        ),
        loc="left",
        fontsize=13,
        fontweight="semibold",
        pad=7,
    )

    ax.set_xlabel(config["xlabel"])
    ax.set_ylabel("Number of subjects")

    ax.set_xlim(*config["xlim"])
    ax.set_xticks(config["xticks"])

    finish_axis(ax)

axes[0].set_ylim(0, 39)
axes[0].set_yticks([0, 8, 16, 24, 32])

fig.subplots_adjust(left=0.065, right=0.985, bottom=0.18, top=0.82, wspace=0.10)

save_figure(fig, "physionet_within_subject_similarity")

plt.show()

#### Figure 15 — PCA projections: original, balanced and extended augmented datasets

In [ ]:
def load_npz_dict(path):
    with np.load(path, allow_pickle=True) as data:
        return {key: data[key] for key in data.files}


def zscore_fit(X):

    mu = np.mean(X, axis=0)

    sigma = np.std(X, axis=0, ddof=1)

    sigma[sigma == 0] = 1.0

    X_z = (X - mu) / sigma

    return X_z, mu, sigma


def zscore_transform(X, mu, sigma):
    return (X - mu) / sigma


real_normal_data = load_npz_dict(
    PIPELINE_DIR / "group_matrices_subject" / "normal_matrix_subject.npz"
)

real_cad_data = load_npz_dict(
    PIPELINE_DIR / "group_matrices_subject" / "cad_matrix_subject.npz"
)

X_normal_real = np.asarray(real_normal_data["X"], dtype=float)

X_cad_real = np.asarray(real_cad_data["X"], dtype=float)

freq_normal_real = np.asarray(real_normal_data["freq_hz"], dtype=float)

freq_cad_real = np.asarray(real_cad_data["freq_hz"], dtype=float)

mixed_normal_data = load_npz_dict(
    PIPELINE_DIR / "group_matrices_mixed_subject" / "normal_matrix_subject.npz"
)

mixed_cad_data = load_npz_dict(
    PIPELINE_DIR / "group_matrices_mixed_subject" / "cad_matrix_subject.npz"
)

X_normal_extended = np.asarray(mixed_normal_data["X"], dtype=float)

X_cad_extended = np.asarray(mixed_cad_data["X"], dtype=float)

is_virtual_normal = np.asarray(mixed_normal_data["is_virtual"], dtype=bool)

is_virtual_cad = np.asarray(mixed_cad_data["is_virtual"], dtype=bool)

freq_normal_mixed = np.asarray(mixed_normal_data["freq_hz"], dtype=float)

freq_cad_mixed = np.asarray(mixed_cad_data["freq_hz"], dtype=float)

X_normal_virtual = X_normal_extended[is_virtual_normal]

X_cad_virtual = X_cad_extended[is_virtual_cad]

X_normal_real_mixed = X_normal_extended[~is_virtual_normal]

X_cad_real_mixed = X_cad_extended[~is_virtual_cad]

N_BALANCED_PER_CLASS = 185

n_virtual_normal_needed = N_BALANCED_PER_CLASS - len(X_normal_real)

n_virtual_cad_needed = N_BALANCED_PER_CLASS - len(X_cad_real)

X_normal_balanced = np.vstack([X_normal_real, X_normal_virtual[:n_virtual_normal_needed]])

X_cad_balanced = np.vstack([X_cad_real, X_cad_virtual[:n_virtual_cad_needed]])

X_original = np.vstack([X_normal_real, X_cad_real])

X_balanced = np.vstack([X_normal_balanced, X_cad_balanced])

X_extended = np.vstack([X_normal_extended, X_cad_extended])

labels_original = np.array(["Normal"] * len(X_normal_real) + ["CAD"] * len(X_cad_real))

labels_balanced = np.array(["Normal"] * len(X_normal_balanced) + ["CAD"] * len(X_cad_balanced))

labels_extended = np.array(["Normal"] * len(X_normal_extended) + ["CAD"] * len(X_cad_extended))

X_original_z, mu_real, sigma_real = zscore_fit(X_original)

X_balanced_z = zscore_transform(X_balanced, mu_real, sigma_real)

X_extended_z = zscore_transform(X_extended, mu_real, sigma_real)

pca_real = PCA(n_components=2, random_state=42)

scores_original = pca_real.fit_transform(X_original_z)

scores_balanced = pca_real.transform(X_balanced_z)

scores_extended = pca_real.transform(X_extended_z)

explained_variance = pca_real.explained_variance_ratio_ * 100

if np.quantile(scores_extended[:, 0], 0.95) < abs(np.quantile(scores_extended[:, 0], 0.05)):
    scores_original[:, 0] *= -1
    scores_balanced[:, 0] *= -1
    scores_extended[:, 0] *= -1

virtual_extended_mask = np.concatenate([is_virtual_normal, is_virtual_cad])

if np.median(scores_extended[virtual_extended_mask, 1]) < 0:
    scores_original[:, 1] *= -1
    scores_balanced[:, 1] *= -1
    scores_extended[:, 1] *= -1

shared_xlim = (-100, 1150)
shared_ylim = (-105, 360)

fig, axes = plt.subplots(1, 3, figsize=(17.7, 4.6), sharex=True, sharey=True)

plot_sets = [
    {
        "scores": scores_original,
        "labels": labels_original,
        "title": (
            "(a) Original subject-level dataset\n"
            f"n = {len(labels_original)}: "
            f"{len(X_normal_real)} Normal, "
            f"{len(X_cad_real)} CAD"
        ),
    },
    {
        "scores": scores_balanced,
        "labels": labels_balanced,
        "title": (
            "(b) Balanced inter-subject augmented dataset\n"
            f"n = {len(labels_balanced)}: "
            f"{len(X_normal_balanced)} Normal, "
            f"{len(X_cad_balanced)} CAD"
        ),
    },
    {
        "scores": scores_extended,
        "labels": labels_extended,
        "title": (
            "(c) Extended augmented dataset\n"
            f"n = {len(labels_extended)}: "
            f"{len(X_normal_extended)} Normal, "
            f"{len(X_cad_extended)} CAD"
        ),
    },
]

for ax, config in zip(axes, plot_sets):
    scores = config["scores"]
    labels = config["labels"]

    for diagnosis, color in [("Normal", PRIMARY_COLOR), ("CAD", SECONDARY_COLOR)]:
        mask = labels == diagnosis

        ax.scatter(
            scores[mask, 0],
            scores[mask, 1],
            s=10,
            color=color,
            alpha=0.72,
            linewidths=0,
            label=diagnosis,
        )

    ax.set_title(config["title"], loc="left", fontsize=13, fontweight="semibold", pad=7)

    ax.set_xlabel(f"PC1 ({explained_variance[0]:.1f}%)")

    ax.set_ylabel(f"PC2 ({explained_variance[1]:.1f}%)")

    ax.set_xlim(*shared_xlim)
    ax.set_ylim(*shared_ylim)

    ax.legend(frameon=False, loc="upper right", fontsize=10, markerscale=1.0)

    finish_axis(ax)

fig.subplots_adjust(left=0.065, right=0.985, bottom=0.17, top=0.80, wspace=0.08)

save_figure(fig, "physionet_pca_original_balanced_extended")

plt.show()

validation_table = pd.DataFrame(
    {
        "Dataset": ["Original", "Balanced", "Extended"],
        "Normal": [len(X_normal_real), len(X_normal_balanced), len(X_normal_extended)],
        "CAD": [len(X_cad_real), len(X_cad_balanced), len(X_cad_extended)],
        "Total": [len(labels_original), len(labels_balanced), len(labels_extended)],
    }
)

display(validation_table)

print(
    "Explained variance:",
    f"PC1 = {explained_variance[0]:.1f}%,",
    f"PC2 = {explained_variance[1]:.1f}%",
)